# Fitting a Hawkes process to data

Everything in the other two notebooks *draws* a realisation. This one goes the
other way: given events, what were the parameters.

The fit is Bayesian and sequential -- a cloud of parameter vectors is reweighted
as each block of data arrives, resampled when the weights concentrate, and moved
by a few Metropolis steps to restore diversity. The likelihood it uses is
computed from the simulator's own intensity hooks, so what is fitted is exactly
what would be drawn.

This notebook is executed on every documentation build, so everything below runs
against the released code. It is budgeted to finish in under a minute.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import hawkes_package as hp
from hawkes_package.inference import (
    ConstrainedPrior,
    ExponentialLogLikelihood,
    HawkesEstimator,
    History,
    IndependentPrior,
    LogNormal,
    RandomWalkDrift,
    SMCSampler,
    SpatioTemporalLogLikelihood,
    Static,
    exponential_model,
    fit_smc,
    ks_exponential,
    posterior_predictive,
    predictive_interval,
    residuals,
    spatio_temporal_model,
)

plt.rcParams["figure.figsize"] = (8, 3.2)
plt.rcParams["figure.dpi"] = 110

## The data

A linear Hawkes process with an exponential kernel, `[mu, alpha, beta] =
[2.0, 0.5, 1.0]`. `History.from_simulation` records the events **and** the window
they were observed on -- here the last event time, which is the correct window
for the output of `simulate(k)`: that call stops at the k-th event, so the
observation really did end there.

In [ ]:
TRUTH = np.array([2.0, 0.5, 1.0])

truth = hp.ExponentialHawkes(TRUTH, rng=7)
truth.simulate(600)
history = History.from_simulation(truth)

print(f"{history.n_events} events on [{history.start}, {history.end:.2f}]")

## The model and the prior

`exponential_model()` is the map from a parameter vector to a process, together
with the set of parameters that map is defined on -- its `support` excludes
`alpha >= beta`, which is exactly what `ExponentialHawkes` refuses to be built
at.

Wrapping the prior in `ConstrainedPrior` with that support keeps the cloud where
the process exists. Without it the sampler raises rather than quietly filtering:
a prior that includes parameters the process cannot be simulated at is worth
correcting.

In [ ]:
model = exponential_model()
prior = ConstrainedPrior(
    IndependentPrior((LogNormal(0.5, 1.0), LogNormal(-1.0, 1.0), LogNormal(0.0, 1.0))),
    model.support,
)

print(model.spec.names)
print(model.support(np.array([[1.0, 0.5, 2.0], [1.0, 3.0, 2.0]])))  # alpha >= beta

## Fitting online

`update` is the entry point: call it as each block arrives. `upto` is the time
the process has been *observed* to, which is not the same as the last event time
-- the difference is the information that nothing happened in between.

The posterior is recorded after every block so the contraction can be plotted.

In [ ]:
smc = SMCSampler(ExponentialLogLikelihood(model), prior, n_particles=256, rng=0)
smc.initialise(start=history.start)

boundaries = np.linspace(history.start, history.end, 13)[1:]
means, lows, highs = [], [], []
for upto in boundaries:
    smc.update(history, float(upto))
    means.append(smc.cloud.mean())
    low, high = smc.cloud.credible_interval(0.9)
    lows.append(low)
    highs.append(high)

means, lows, highs = np.array(means), np.array(lows), np.array(highs)
print(smc.cloud.summary())

### The posterior contracts, and it contains the truth

Each panel is one parameter: the posterior mean, its 90% credible band, and the
value the data was generated at. The bands narrow as data arrives, which is what
learning looks like -- and they keep containing the dashed line, which is what
being *right* looks like.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for j, (ax, name) in enumerate(zip(axes, model.spec.names, strict=True)):
    ax.fill_between(boundaries, lows[:, j], highs[:, j], alpha=0.25, label="90% interval")
    ax.plot(boundaries, means[:, j], lw=2, label="posterior mean")
    ax.axhline(TRUTH[j], ls="--", color="k", lw=1, label="truth")
    ax.set_title(name)
    ax.set_xlabel("observed to")
axes[0].legend(fontsize=8)
fig.tight_layout()

## Reading the diagnostics

A posterior can be tight, stable and wrong, so the fit reports how it got there.

`ESS` is measured *before* each resample, and a low value is not a fault -- it
says the block was informative. What matters is that a resample followed, and
that the move afterwards actually moved the cloud. `move` is that distance, in
units of the cloud's own width; the acceptance rate cannot answer it, because a
proposal scaled to nothing proposes the point it starts from and is accepted
almost always.

`warnings()` reports the combinations that mean something. An empty list is the
thing to check for.

In [ ]:
print(smc.diagnostics.summary())
print()
print("warnings:", smc.diagnostics.warnings() or "none")

## Does the model fit?

The time-rescaling theorem: mapping the events through the compensator at the
fitted parameter should give a unit-rate Poisson process. A quantile plot of the
rescaled gaps against `Exp(1)` should sit on the diagonal, and a
Kolmogorov-Smirnov test puts a number on it.

This single check catches a wrong kernel, a missing background and an
under-counted compensator at once -- none of which the plot above would notice.

In [ ]:
likelihood = ExponentialLogLikelihood(model)
gaps = residuals(likelihood, smc.cloud.mean(), history)
result = ks_exponential(gaps)

quantiles = -np.log1p(-(np.arange(1, gaps.size + 1) - 0.5) / gaps.size)
fig, ax = plt.subplots()
ax.plot(quantiles, np.sort(gaps), ".", ms=4)
ax.plot([0, quantiles.max()], [0, quantiles.max()], "k--", lw=1)
ax.set_xlabel("Exp(1) quantile")
ax.set_ylabel("rescaled gap")
ax.set_title(f"time-rescaling: D={result.statistic:.3f}, p={result.pvalue:.3f}")
fig.tight_layout()

## The same fit in one object

Everything above assembled four objects by hand -- a model, a prior, a
likelihood and a sampler -- because that is what the fit *is*. `HawkesEstimator`
holds all four behind the method names a user of scikit-learn already knows, and
infers nothing of its own: a `fit` reproduces `fit_smc` bit-for-bit at the same
seed, and a `partial_fit` per block reproduces `fit(blocks=k)` bit-for-bit.

It does not import scikit-learn, and does not need it installed.

In [ ]:
# 128 particles rather than the 256 above: this section is showing the interface,
# not re-deriving the posterior, and `predict` below costs one process build per
# particle.
est = HawkesEstimator(model, prior, n_particles=128, blocks=8, rng=0).fit(history)

print("posterior mean", est.theta_.round(3), " truth", TRUTH)
print("warnings:", est.diagnostics_.warnings() or "none")
print()
print(est.report(truth=TRUTH))

### The intensity, with the posterior carried through

`predict` returns the conditional intensity at the times you ask for. It
**averages over the particles** rather than plugging in `theta_`: the intensity
is convex in the decay rate, so the plug-in sits systematically below the
marginal wherever the posterior has width. `predict_intensity_band` is the
spread that mean is taken over, and it is the reason the marginalisation is
worth its cost.

It **refuses** times past `history.end`. There the intensity computed from the
observed record is the intensity given that nothing has happened since, which
understates the truth by exactly the excitation of the events that would have
occurred -- the next section forecasts properly instead, by simulating forward.

In [ ]:
# A window rather than the whole span: 600 events on one axis is an unreadable
# smear, and `predict` is O(n_particles x len(grid) x n_events), so a 300-point
# window is also what keeps this cell around a second.
grid = np.linspace(history.start, 15.0, 300)
band_low, _, band_high = est.predict_intensity_band(grid, level=0.9)
shown = history.times[history.times <= grid[-1]]

fig, ax = plt.subplots()
ax.fill_between(grid, band_low, band_high, alpha=0.25, label="90% posterior band")
ax.plot(grid, est.predict(grid), lw=2, label="posterior mean intensity")
ax.plot(shown, np.zeros(shown.size), "|", ms=9, color="k", label="events")
ax.axhline(TRUTH[0], ls="--", color="grey", lw=1, label=r"true $\mu$")
ax.set_xlabel("time")
ax.set_ylabel(r"$\lambda(t \mid H_t)$")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()

`score` is the log posterior-predictive density of the window that comes *next*,
so it needs a fit that stopped short of the end. Higher is better, and it is the
same quantity the evidence accumulates -- scoring a block and then absorbing it
with `partial_fit` gives the number twice.

In [ ]:
cut = 0.8 * history.end
early = HawkesEstimator(model, prior, n_particles=128, blocks=6, rng=0).fit(history.upto(cut))
tail = history.times[history.times > cut]

print(f"fitted on [0, {cut:.1f}], {early.n_events_} events")
print(
    f"log predictive density of the remaining {history.end - cut:.1f} time units: "
    f"{early.score(tail, end=history.end):.2f}"
)

## Forecasting

Each predictive path draws a fresh particle, so the band carries the uncertainty
about *which* process as well as the process's own variability. Simulating from
the posterior mean instead would give a band that is too narrow.

The paths start at `history.end` rather than at the last event. The gap between
those is data -- it says nothing happened there -- and this is what
`simulate_until(..., start=)` exists for.

In [ ]:
paths = posterior_predictive(model, smc.cloud, history, horizon=25.0, n_paths=200, rng=1)
grid = np.linspace(history.end, history.end + 25.0, 60)
low, mid, high = predictive_interval(paths, grid)

fig, ax = plt.subplots()
ax.fill_between(grid, low, high, alpha=0.25, label="90% predictive band")
ax.plot(grid, mid, lw=2, label="median")
for path in paths[:12]:
    ax.step(np.concatenate([[history.end], path]), np.arange(path.size + 1), lw=0.6, alpha=0.5)
ax.set_xlabel("time")
ax.set_ylabel("events since the window closed")
ax.legend(fontsize=8)
fig.tight_layout()

## Space as well as time

The same four objects with a domain attached. `spatio_temporal_model` gives
`(mu, alpha, beta, sigma)` -- background per unit length, temporal excitation and
decay, and the spatial scale.

The `backend` is the thing to watch. `"hooks"` is the definition, and calls the
same `_integrated_intensity` the simulator thins against; it is also several
orders of magnitude too slow to fit with. `"cached"` rearranges the same quantity
around the separability of the intensity, precomputing the distances that do not
depend on the parameters. It is asked for **explicitly** here, so a silent
fallback would blow this notebook's time budget visibly rather than quietly.

In [ ]:
st_truth = np.array([0.5, 0.6, 1.5, 0.5])
st_model = spatio_temporal_model(hp.Circle(), n_quad=64)

st_process = st_model(st_truth, rng=31)
st_process.simulate(40)
st_history = History.from_simulation(st_process)

st_likelihood = SpatioTemporalLogLikelihood(st_model, backend="cached")
st_smc = fit_smc(
    st_likelihood,
    ConstrainedPrior(
        IndependentPrior(
            (
                LogNormal(-0.7, 0.8),
                LogNormal(-0.5, 0.8),
                LogNormal(0.4, 0.8),
                LogNormal(-0.7, 0.6),
            )
        ),
        st_model.support,
    ),
    st_history,
    blocks=4,
    n_particles=64,
    rng=5,
)

print(f"backend={st_likelihood.backend_used}, spread={st_likelihood.spatial_spread:.2e}")
print(st_smc.cloud.summary())
print("truth      ", st_truth)

In [ ]:
fig, ax = plt.subplots()
ax.scatter(st_history.times, st_history.points[0], s=14)
ax.set_xlabel("time")
ax.set_ylabel("position on the circle")
ax.set_ylim(-np.pi, np.pi)
ax.set_title("the events the fit above was given")
fig.tight_layout()

## When the parameter moves

Everything so far assumed one fixed parameter the data is evidence about -- and
under that assumption the increments telescope, which is what makes the online
fit exact. If the parameter itself *moves*, that assumption is wrong, and a
static fit does not merely lag: it has no mechanism to move at all, because a
cloud that has contracted has nothing left to re-expand it.

`evolution=RandomWalkDrift(...)` jitters the particles between blocks, on the
unconstrained scale, and buys back the diversity. `n_move=0` is then required
rather than optional: an MCMC move is invariant for a target the model has just
said no longer exists.

The data below changes regime. It is built by simulating to the switch, seeding
a second process with that history, and continuing -- which is what
`simulate_until(..., start=)` is for, since `start` asserts that nothing happened
between the last recorded event and the moment the second regime begins.

In [ ]:
switch, end, n_blocks = 40.0, 80.0, 40

quiet = hp.ExponentialHawkes(np.array([1.0, 0.05, 2.0]), rng=20)
quiet.simulate_until(switch)

busy = hp.ExponentialHawkes(np.array([5.0, 0.05, 2.0]), rng=40)
busy.events = quiet.events  # seed the second regime with the first regime's history
busy.simulate_until(end, start=switch)

drift_history = History.from_events(busy.events, start=0.0, end=end)
drift_prior = ConstrainedPrior(
    IndependentPrior((LogNormal(0.8, 1.0), LogNormal(-2.0, 1.0), LogNormal(0.0, 1.0))),
    model.support,
)
mu_index = model.spec.index("mu")
edges = np.linspace(0.0, end, n_blocks + 1)[1:]


def track_mu(evolution, n_move):
    """Fit the same history block by block, recording the posterior mean of mu."""
    smc = SMCSampler(
        ExponentialLogLikelihood(model),
        drift_prior,
        n_particles=128,
        evolution=evolution,
        n_move=n_move,
        rng=3,
    )
    smc.initialise(start=drift_history.start)
    trace = []
    for upto in edges:
        smc.update(drift_history, float(upto))
        trace.append(float(smc.cloud.mean()[mu_index]))
    return np.array(trace)


drifting = track_mu(RandomWalkDrift(0.15), 0)
unmoving = track_mu(Static(), 3)

before = int((drift_history.times <= switch).sum())
print(f"{drift_history.n_events} events, {before} of them before the switch")

In [ ]:
fig, ax = plt.subplots()
ax.step(
    [0.0, switch, end], [1.0, 5.0, 5.0], where="post", ls="--", color="k", lw=1, label=r"true $\mu$"
)
ax.plot(edges, drifting, lw=2, label="RandomWalkDrift(0.15)")
ax.plot(edges, unmoving, lw=2, label="Static()")
ax.axvline(switch, color="grey", lw=1, alpha=0.6)
ax.set_xlabel("observed to")
ax.set_ylabel(r"posterior mean of $\mu$")
ax.legend(fontsize=8)
fig.tight_layout()

The drifting filter climbs after the switch; the static one *falls*, because the
extra events are evidence against the small background it had already settled on
and shrinking is the only way it can reconcile them. The climb also visibly lags
the true step, and that is not a tuning failure: the block likelihood applies one
parameter to a whole block, including the excitation from events generated under
the previous one, so the approximation costs exactly this.

`LiuWest` is the other shipped kernel, and it cannot do this at all. Its jitter
is proportional to the cloud's own variance, so once the cloud has contracted
there is nothing left to re-expand it -- it handles a parameter that is merely
*uncertain*, not one that moves.

## Where to go next

- [Fitting a process to data](../inference.md) -- choosing a prior, reading the
  diagnostics, drifting parameters, and what each path costs.
- [Theory](../theory.md#inference) -- why a bootstrap filter is the wrong
  algorithm here, what the compensator quadrature has to resolve, and what the
  drifting-parameter approximation costs.